In [7]:
# CELL 1: Install required packages (run in notebook environment)
# You may already have some packages. Uncomment installs if needed.

# !pip install mne numpy scipy matplotlib scikit-learn tqdm torch torchvision torchaudio pytorch-lightning datasetlib
# If TQWT implementation library not available, we implement a lightweight TQWT below.

import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal
from scipy import stats
from scipy.fftpack import fft
from scipy.signal import welch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, auc, confusion_matrix,
                             classification_report, accuracy_score)
from sklearn.manifold import TSNE
import joblib
import math
import pickle
# set matplotlib inline if using classic jupyter
# %matplotlib inline


In [8]:
# CELL 2: configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

datasetPath = r"E:\EEG\eeg"                # <-- user-specified path
labels_csv_path = r"E:\EEG\eeg\derivatives\labels.csv"  # provided by user

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda:0


In [9]:
# CELL 3: feature extraction helpers

def tjwt_decompose(signal_in, Q=1.0, r=10, J=6):
    """
    Lightweight TQWT-like decomposition approximation.
    This is a simplified implementation that uses DWT-like iterative bandpass splitting.
    For production, replace with validated TQWT implementation.
    Returns list of subband arrays (length J+1).
    """
    # placeholder: split with iterative Q-factor-like IIR filters (approx)
    # We'll implement a bank of bandpass filters using octave-like splits.
    fs = 500.0
    x = signal_in
    bands = []
    low = 0.5
    high = fs/2.0
    # compute logarithmic band edges for J subbands
    freqs = np.geomspace(low+1e-6, high, num=J+2)
    for i in range(J+1):
        f1 = freqs[i]
        f2 = freqs[i+1]
        b, a = signal.butter(4, [f1/(fs/2.0), min(f2/(fs/2.0), 0.999)], btype='band')
        y = signal.filtfilt(b, a, x)
        bands.append(y)
    return bands

def bandpower(x, fs, band):
    fmin, fmax = band
    f, Pxx = welch(x, fs=fs, nperseg=1024)
    mask = (f >= fmin) & (f <= fmax)
    return np.trapz(Pxx[mask], f[mask])

def spectral_entropy(x, fs):
    f, Pxx = welch(x, fs=fs, nperseg=1024)
    P = Pxx / np.sum(Pxx)
    P = P + 1e-12
    return -np.sum(P * np.log2(P))

def dominant_freq(x, fs):
    f, Pxx = welch(x, fs=fs, nperseg=1024)
    return f[np.argmax(Pxx)]

def hjorth_params(x):
    dx = np.diff(x)
    ddx = np.diff(dx)
    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)
    activity = var_x
    mobility = np.sqrt(var_dx / (var_x + 1e-12))
    complexity = np.sqrt((var_ddx / (var_dx + 1e-12)) / (var_dx / (var_x + 1e-12)) + 1e-12)
    return activity, mobility, complexity

def tkeo(x):
    # Teager-Kaiser Energy Operator
    x = np.asarray(x)
    return np.mean(x[1:-1]**2 - x[:-2]*x[2:])


In [10]:
# CELL 4: windowing + feature extraction for a single raw signal array (n_channels x n_samples)

FS = 500
WINDOW_SEC = 5.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)          # 2500
STEP_SAMPLES = WINDOW_SAMPLES // 2             # 50% overlap -> 1250

# Frequency bands
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 7),
    'alpha': (8, 12),
    'beta': (13, 30)
}

def extract_features_from_window(win_data, fs=FS, tqwt_params=(1.0,10,6)):
    """
    win_data: ndarray shape (n_channels, n_samples)
    returns: ndarray shape (n_channels, n_features_per_channel)
    """
    n_ch = win_data.shape[0]
    feats = []
    Q, R, J = tqwt_params
    for ch in range(n_ch):
        x = win_data[ch, :]
        # time-domain
        mean_ = np.mean(x)
        rms = np.sqrt(np.mean(x**2))
        activity, mobility, complexity = hjorth_params(x)
        tkeo_v = tkeo(x)
        # freq-domain
        band_pows = [bandpower(x, fs, BANDS[b]) for b in ['delta','theta','alpha','beta']]
        spec_ent = spectral_entropy(x, fs)
        domf = dominant_freq(x, fs)
        # TQWT approx
        bands = tjwt_decompose(x, Q=Q, r=R, J=J)
        tqwt_feats = []
        for b in bands:
            tqwt_feats.extend([np.mean(b), np.std(b), np.sum(b**2)])  # mean, std, energy
        # gather
        ch_feats = [mean_, rms, activity, mobility, complexity, tkeo_v] + band_pows + [spec_ent, domf] + tqwt_feats
        feats.append(ch_feats)
    feats = np.array(feats)  # (n_ch, n_feat_per_channel)
    return feats

def sliding_windows_and_features(raw_data, fs=FS, window_samples=WINDOW_SAMPLES, step=STEP_SAMPLES, tqwt_params=(1.0,10,6)):
    """
    raw_data: ndarray shape (n_channels, n_samples)
    returns: features array shape (n_windows, n_channels, n_features_per_channel)
    """
    n_samples = raw_data.shape[1]
    idx = 0
    feats_list = []
    while (idx + window_samples) <= n_samples:
        win = raw_data[:, idx:idx+window_samples]
        feats = extract_features_from_window(win, fs=fs, tqwt_params=tqwt_params)
        feats_list.append(feats)
        idx += step
    if len(feats_list) == 0:
        return np.empty((0, raw_data.shape[0], 0))
    return np.stack(feats_list, axis=0)


In [11]:
# CELL 5: load labels.csv, select sub-001 -> sub-065, AD + CN only, build features cache

labels_df = pd.read_csv(labels_csv_path)
# Ensure subject column naming match, standardize
labels_df.columns = [c.strip() for c in labels_df.columns]
if 'subject' not in labels_df.columns:
    # assume first column is subject
    labels_df.rename(columns={labels_df.columns[0]:'subject', labels_df.columns[1]:'label'}, inplace=True)

# Filter subjects sub-001 .. sub-065
sel = labels_df['subject'].str.startswith('sub-') & (labels_df['subject'].str.slice(4).astype(int) <= 65)
labels_sel = labels_df[sel].copy()
labels_sel = labels_sel[labels_sel['label'].isin(['AD','CN','Control','HC'])]
# map variants to AD / CN
labels_sel['label'] = labels_sel['label'].replace({'Control':'CN','HC':'CN'})

print("Selected subjects:", len(labels_sel))
print(labels_sel['label'].value_counts())

# Function to load raw EEG for a subject using MNE (supports many BIDS/raw formats)
def load_subject_raw(subject_id, dataset_root=datasetPath):
    """
    Tries common BIDS paths and file types (.edf, .fif, .set). Adjust if your files differ.
    Returns raw_data numpy array shape (n_channels, n_samples) and channel names
    """
    subj_dir = Path(dataset_root) / f"{subject_id}"
    # Try scanning for common EEG files recursively
    candidates = list(subj_dir.rglob("*.edf")) + list(subj_dir.rglob("*.set")) + list(subj_dir.rglob("*.fif")) + list(subj_dir.rglob("*.bdf"))
    if len(candidates) == 0:
        raise FileNotFoundError(f"No EEG files found for {subject_id} in {subj_dir}")
    fp = str(candidates[0])
    print("Loading", fp)
    raw = None
    if fp.endswith('.edf') or fp.endswith('.bdf'):
        raw = mne.io.read_raw_edf(fp, preload=True, verbose=False)
    elif fp.endswith('.fif'):
        raw = mne.io.read_raw_fif(fp, preload=True, verbose=False)
    elif fp.endswith('.set'):
        raw = mne.io.read_raw_eeglab(fp, preload=True, verbose=False)
    else:
        raise RuntimeError("Unknown format: " + fp)
    # pick the 19 channels if present
    picks = [ch for ch in raw.ch_names if ch.upper() in ['FP1','FP2','F7','F3','FZ','F4','F8','T3','C3','CZ','C4','T4','T5','P3','PZ','P4','T6','O1','O2']]
    raw.pick_channels(picks)
    raw.load_data()
    data = raw.get_data()  # shape (n_channels, n_samples)
    return data, raw.ch_names

# Cache features to disk to avoid reprocessing
cache_dir = Path(datasetPath) / "feature_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

subject_feature_meta = []  # list of dicts: {subject, label, features_path, n_windows}
for _, row in labels_sel.iterrows():
    subj = row['subject']
    lab = 'AD' if row['label']=='AD' else 'CN'
    cache_file = cache_dir / f"{subj}_features.pkl"
    if cache_file.exists():
        with open(cache_file,'rb') as f:
            feats = pickle.load(f)
    else:
        raw_data, ch_names = load_subject_raw(subj, datasetPath)
        # ensure sampling rate 500; if different, resample
        # raw.info['sfreq'] accessible if using mne, but here raw_data assumed 500 per spec
        feats = sliding_windows_and_features(raw_data, fs=FS, window_samples=WINDOW_SAMPLES, step=STEP_SAMPLES, tqwt_params=(1.0,10,6))
        with open(cache_file,'wb') as f:
            pickle.dump({'features':feats, 'ch_names': ch_names}, f)
    n_windows = feats['features'].shape[0] if isinstance(feats, dict) else feats.shape[0]
    subject_feature_meta.append({'subject':subj, 'label':lab, 'cache':str(cache_file), 'n_windows':n_windows})

subject_feature_meta = pd.DataFrame(subject_feature_meta)
print(subject_feature_meta.head())


Selected subjects: 65
label
AD    36
CN    29
Name: count, dtype: int64
Loading E:\EEG\eeg\sub-001\eeg\sub-001_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


C:\Users\sanja\AppData\Local\Temp\ipykernel_14640\104017106.py:31: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(Pxx[mask], f[mask])


Loading E:\EEG\eeg\sub-002\eeg\sub-002_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading E:\EEG\eeg\sub-003\eeg\sub-003_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading E:\EEG\eeg\sub-004\eeg\sub-004_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading E:\EEG\eeg\sub-005\eeg\sub-005_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading E:\EEG\eeg\sub-006\eeg\sub-006_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading E:\EEG\eeg\sub-007\eeg\sub-007_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loading E:\EEG\eeg\sub-008\eeg\sub-008_task-eyesclosed_eeg.set
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

In [12]:
# CELL 6: PyTorch Dataset that returns sequences of windows per subject
class EEGWindowDataset(Dataset):
    """
    Returns one window (features tensor) per sample.
    Each sample: features shaped (n_channels, n_features_per_channel) flattened or left as is.
    """
    def __init__(self, meta_df, scaler=None, channel_first=True):
        self.meta = meta_df.reset_index(drop=True)
        self.records = []
        for idx, row in self.meta.iterrows():
            cache_file = row['cache']
            with open(cache_file, 'rb') as f:
                data = pickle.load(f)
            feats = data['features']  # (n_windows, n_channels, n_feat_per_ch)
            # flatten per channel: keep channel x features
            # We'll flatten channels into feature-dim for model input: (n_windows, n_channels * n_feat)
            n_windows, n_ch, n_feat = feats.shape
            feats_reshaped = feats.reshape(n_windows, n_ch * n_feat)
            for w in range(n_windows):
                self.records.append({
                    'subject': row['subject'],
                    'label': 1 if row['label']=='AD' else 0,
                    'features': feats_reshaped[w]
                })
        self.X = np.stack([r['features'] for r in self.records], axis=0).astype(np.float32)
        self.y = np.array([r['label'] for r in self.records], dtype=np.int64)
        # scaling
        if scaler is None:
            self.scaler = StandardScaler().fit(self.X)
        else:
            self.scaler = scaler
        self.X = self.scaler.transform(self.X)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        # convert to tensors
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# Create a global scaler from all data (optional caching)
all_meta = subject_feature_meta.copy()
dataset_all = EEGWindowDataset(all_meta)
SCALER = dataset_all.scaler
print("Total windows:", len(dataset_all), "Feature dim:", dataset_all.X.shape[1])


C:\Users\sanja\AppData\Local\Temp\ipykernel_14640\3241849402.py:25: RuntimeWarning: overflow encountered in cast
  self.X = np.stack([r['features'] for r in self.records], axis=0).astype(np.float32)


ValueError: Input X contains infinity or a value too large for dtype('float32').

In [ ]:
# CELL 7: models

class EEGTransformer(nn.Module):
    def __init__(self, input_dim, d_model=256, nhead=8, num_encoder_layers=3, dim_feedforward=512, dropout=0.2, num_classes=2):
        super().__init__()
        self.input_dim = input_dim
        # project input features to d_model
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward, dropout=dropout,
                                                   batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model//2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_model//2, num_classes)
        )
    def forward(self, x):
        # x shape (batch, features) -> treat features as "sequence length 1" of a token embedding
        # alternative: reshape to (batch, seq_len, feat) if sequence provided
        # We'll project into d_model and add a learnable CLS token behavior by unsqueezing length=1
        x = self.input_proj(x)        # (batch, d_model)
        x = x.unsqueeze(1)           # (batch, seq=1, d_model)
        # transformer expects seq>0; we still pass seq=1 through encoder layers
        t_out = self.transformer(x)  # (batch, 1, d_model)
        cls = t_out.mean(dim=1)      # (batch, d_model)
        out = self.classifier(cls)
        return out

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_size=256, num_layers=2, bidirectional=True, dropout=0.3, num_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_size, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional, dropout=dropout)
        fc_in = hidden_size * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Linear(fc_in, fc_in//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_in//2, num_classes)
        )
    def forward(self, x):
        # x: (batch, features) -> treat as seq_len=1
        x = x.unsqueeze(1)  # (batch, seq=1, feat)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.classifier(last)


In [ ]:
# CELL 8: training utilities

def mixup_data(x, y, alpha=0.4, device=DEVICE):
    if alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

class EarlyStopping:
    def __init__(self, patience=10, delta=1e-4, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_score = None
        self.counter = 0
        self.early_stop = False
    def step(self, val):
        score = val
        if self.best_score is None:
            self.best_score = score
            return False
        if score <= self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
            return False
        else:
            self.best_score = score
            self.counter = 0
            return False

def train_one_epoch(model, dataloader, optimizer, criterion, device=DEVICE, mixup_alpha=0.4):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(dataloader, desc="Train batches", leave=False)
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        if mixup_alpha and mixup_alpha>0:
            xb, ya, yb_shuf, lam = mixup_data(xb, yb, alpha=mixup_alpha, device=device)
            outputs = model(xb)
            loss = lam * criterion(outputs, ya) + (1-lam) * criterion(outputs, yb_shuf)
        else:
            outputs = model(xb)
            loss = criterion(outputs, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        preds = outputs.argmax(dim=1)
        # for mixup, accuracy isn't exact but we still measure against original labels (approx)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
        pbar.set_postfix(loss=running_loss/total, acc=correct/total)
    return running_loss/total, correct/total

def eval_model(model, dataloader, device=DEVICE):
    model.eval()
    ys = []
    probs = []
    preds = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            outs = model(xb)
            p = F.softmax(outs, dim=1)[:,1].detach().cpu().numpy()
            pr = outs.argmax(dim=1).detach().cpu().numpy()
            ys.extend(yb.numpy().tolist())
            probs.extend(p.tolist())
            preds.extend(pr.tolist())
    return np.array(ys), np.array(probs), np.array(preds)


In [ ]:
# CELL 9: 10-fold subject-wise cross-validation training & evaluation

NUM_FOLDS = 10
BATCH_SIZE = 64
EPOCHS = 80
PATIENCE = 12
LR = 1e-4
MODEL_SAVE_DIR = Path(datasetPath) / "models"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Build per-subject labels for CV splitting
# We need a subject-wise label vector for only AD/CN selected subjects
cv_meta = subject_feature_meta.copy()
y_sub = cv_meta['label'].map({'AD':1,'CN':0}).values
subjects = cv_meta['subject'].values

skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(subjects, y_sub), 1):
    print(f"\n=== Fold {fold}/{NUM_FOLDS} ===")
    train_subjects = cv_meta.iloc[train_idx]
    val_subjects = cv_meta.iloc[val_idx]
    # create dataset instances that load windows for subjects in train/val
    train_dataset = EEGWindowDataset(train_subjects, scaler=SCALER)
    val_dataset = EEGWindowDataset(val_subjects, scaler=SCALER)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    input_dim = train_dataset.X.shape[1]
    # Switch model here: EEGTransformer or LSTMClassifier
    model = EEGTransformer(input_dim=input_dim, d_model=256, nhead=8, num_encoder_layers=3, dropout=0.2, num_classes=2).to(DEVICE)
    # Compute class weights from train_dataset
    class_counts = np.bincount(train_dataset.y)
    class_weights = torch.tensor([ (class_counts.sum() / (2.0 * c + 1e-12)) for c in class_counts ], dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4, verbose=True)
    earlyst = EarlyStopping(patience=PATIENCE)
    best_val_auc = 0.0
    best_model_path = MODEL_SAVE_DIR / f"eeg_transformer_fold{fold}.pt"
    history = {'train_loss':[], 'train_acc':[], 'val_auc':[], 'val_loss':[]}
    for epoch in range(1, EPOCHS+1):
        print(f"Epoch {epoch}/{EPOCHS}")
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device=DEVICE, mixup_alpha=0.3)
        ys_val, probs_val, preds_val = eval_model(model, val_loader, device=DEVICE)
        val_auc = roc_auc_score(ys_val, probs_val) if len(np.unique(ys_val))>1 else 0.5
        val_loss = F.cross_entropy(torch.tensor(preds_val).float(), torch.tensor(ys_val).long()).item() if len(preds_val)>0 else 0.0
        print(f"Epoch {epoch} Train loss {train_loss:.4f} acc {train_acc:.4f} | Val AUC {val_auc:.4f}")
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_auc'].append(val_auc)
        history['val_loss'].append(val_loss)
        scheduler.step(val_auc)
        # save best
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save({'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scaler': SCALER}, best_model_path)
            print("Saved best model:", best_model_path)
        # early stop check (based on val_auc)
        if earlyst.step(val_auc):
            print("Early stopping triggered.")
            break
    # After fold end, evaluate best ckpt
    ckpt = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    ys_val, probs_val, preds_val = eval_model(model, val_loader, device=DEVICE)
    acc = accuracy_score(ys_val, preds_val)
    auc_score = roc_auc_score(ys_val, probs_val) if len(np.unique(ys_val))>1 else 0.5
    cr = classification_report(ys_val, preds_val, target_names=['CN','AD'], zero_division=0)
    print("Fold", fold, "ACC:", acc, "AUC:", auc_score)
    print(cr)
    fold_results.append({'fold':fold, 'acc':acc, 'auc':auc_score, 'report':cr, 'history':history})


In [ ]:
# CELL 10: plotting utilities & example plots from last fold
import matplotlib.pyplot as plt
last = fold_results[-1]
h = last['history']
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(h['train_loss'], label='train_loss')
plt.plot(h['val_loss'], label='val_loss')
plt.legend(); plt.title("Loss curves")
plt.subplot(1,2,2)
plt.plot(h['train_acc'], label='train_acc')
plt.plot(h['val_auc'], label='val_auc')
plt.legend(); plt.title("Acc / AUC")
plt.show()

# ROC curve
from sklearn.metrics import roc_curve
ys_val, probs_val, preds_val = ys_val, probs_val, preds_val
fpr, tpr, _ = roc_curve(ys_val, probs_val)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC={auc(fpr,tpr):.3f}")
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC"); plt.legend()
plt.show()

# Confusion matrix
cm = confusion_matrix(ys_val, preds_val)
plt.figure(figsize=(4,4))
plt.imshow(cm, cmap='Blues')
plt.title("Confusion matrix")
plt.colorbar()
plt.xticks([0,1], ['CN','AD'])
plt.yticks([0,1], ['CN','AD'])
for i in range(2):
    for j in range(2):
        plt.text(j,i,cm[i,j],ha='center',va='center',color='black')
plt.show()


In [ ]:

# CELL 11: t-SNE visualization on a subset
X = dataset_all.X
y = dataset_all.y
sel_idx = np.random.choice(np.arange(len(y)), size=min(2000, len(y)), replace=False)
X_sub = X[sel_idx]
y_sub = y[sel_idx]
ts = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
X_emb = ts.fit_transform(X_sub)
plt.figure(figsize=(7,5))
plt.scatter(X_emb[y_sub==0,0], X_emb[y_sub==0,1], s=6, label='CN')
plt.scatter(X_emb[y_sub==1,0], X_emb[y_sub==1,1], s=6, label='AD')
plt.legend(); plt.title("t-SNE (feature space)")
plt.show()
